# 02 - Statistical Baseline (safety net)

Flattens **Bohra 2018 + HASOC 2021 + HASOC 2022** into one frame `[text, label, source, script]`,
filters to **romanised** rows, and runs a **TF-IDF + Logistic Regression** baseline.

Outputs:
1. **within-Bohra macro-F1** - the safety-net number.
2. **cross-dataset macro-F1** (train Bohra, test HASOC) - the generalisation drop.

HASOC 2022 is loaded from the **per-tweet thread files** (`data.json` + `binary_labels.json`),
not `final.csv`, so each row is a single utterance comparable to Bohra. The appendix shows why.

Runs top-to-bottom on a fresh kernel. Put `hinglish_hate/` next to this notebook and set `DATA_ROOT`.

### 1. Mount Drive

In [6]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not on Colab / already mounted:', e)

Not on Colab / already mounted: No module named 'google.colab'


### 2. Import the package

In [7]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')

from hinglish_hate import (
    load_bohra, load_hasoc2021, load_hasoc2022,
    build_corpus, filter_romanised,
    evaluate_within, evaluate_cross, summarise,
)
from hinglish_hate.loaders import _walk_thread, _to_binary, _finish
import pandas as pd, json
from pathlib import Path
print('package imported OK')

package imported OK


### 3. Locate the data files

In [8]:
# DATA_ROOT = Path('/content/drive/MyDrive/dissertation/data')
DATA_ROOT = Path(r'C:\Users\LENOVO\Desktop\dissertation\data')

def find_one(root, name):
    hits = list(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f'{name} not found under {root}')
    return hits[0]

bohra_path = find_one(DATA_ROOT, 'hate_speech.tsv')
h21_labels = list(Path(DATA_ROOT).rglob('labels.json'))
h21_root   = h21_labels[0].parents[2] if h21_labels else None
print('Bohra   :', bohra_path)
print('HASOC21 :', h21_root, f'({len(h21_labels)} thread label files)')

Bohra   : C:\Users\LENOVO\Desktop\dissertation\data\hate_speech.tsv
HASOC21 : C:\Users\LENOVO\Desktop\dissertation\data\test-20220220T075607Z-001\test (57 thread label files)


### 4. Clean HASOC 2022 loader

Walks the 2022 Hinglish thread folders and joins each tweet's own text to its binary label -
single utterances, not accumulated thread context. (Fold into `loaders.py` when convenient;
`01_data_exploration` uses the same function.)

In [9]:
def load_hasoc2022_threads(root):
    root = Path(root)
    thread_dirs = {p.parent for p in root.rglob('binary_labels.json')
                   if (p.parent / 'data.json').exists()}
    rows = []
    for d in sorted(thread_dirs):
        data   = json.load(open(d / 'data.json', encoding='utf-8'))
        labels = json.load(open(d / 'binary_labels.json', encoding='utf-8'))
        id2text = {}
        _walk_thread(data, id2text)
        for tid, lab in labels.items():
            t = id2text.get(str(tid))
            if t:
                rows.append((t, _to_binary(lab)))
    return _finish(rows, 'hasoc2022')

### 5. Load and profile each source

Mirrors the label/script breakdown from `01_data_exploration`.

In [10]:
bohra = load_bohra(bohra_path)
h21   = load_hasoc2021(h21_root) if h21_root else None
h22   = load_hasoc2022_threads(DATA_ROOT)   # clean per-tweet HASOC22 - canonical

for name, df in [('bohra2018', bohra), ('hasoc2021', h21), ('hasoc2022', h22)]:
    if df is None:
        print(f'{name:11s} - skipped'); continue
    print(f"{name:11s} rows={len(df):5d}  hate%={df['label'].mean():.2f}")
    print('   scripts:', dict(df['script'].value_counts()))

[load_bohra] 4574 rows | fixed 0 label typo(s) | dropped 4 unparseable
bohra2018   rows= 4574  hate%=0.36
   scripts: {'mostly_latin': np.int64(4574)}
hasoc2021   rows= 2092  hate%=0.50
   scripts: {'mostly_latin': np.int64(1687), 'mostly_devanagari': np.int64(261), 'mixed_script': np.int64(144)}
hasoc2022   rows= 4901  hate%=0.51
   scripts: {'mostly_latin': np.int64(3998), 'mostly_devanagari': np.int64(595), 'mixed_script': np.int64(308)}


### 6. Unify + filter to romanised

In [11]:
frames = [f for f in [bohra, h21, h22] if f is not None]
corpus = build_corpus(frames)
roman  = filter_romanised(corpus, include_mixed=True)

print('full corpus  :', len(corpus))
print('romanised    :', len(roman))
print()
print(pd.crosstab(roman['source'], roman['label']).rename(columns={0:'not', 1:'hate'}))

full corpus  : 11548
romanised    : 10694

label       not  hate
source               
bohra2018  2914  1660
hasoc2021   898   926
hasoc2022  2092  2204


## 7. Safety-net baseline (within Bohra)

**The number to report first.** Stratified 80/20 split on romanised Bohra, TF-IDF + Logistic
Regression, macro-F1.

In [12]:
bohra_r = filter_romanised(bohra)  # Latin-only
within = evaluate_within(bohra_r)
print(summarise('within Bohra', within))
print()
print(within['report'])

[within Bohra]  macro-F1 = 0.629   hate-F1 = 0.544   acc = 0.648   (train 3659, test 915)

              precision    recall  f1-score   support

         not      0.741     0.688     0.714       583
        hate      0.513     0.578     0.544       332

    accuracy                          0.648       915
   macro avg      0.627     0.633     0.629       915
weighted avg      0.659     0.648     0.652       915



## 8. Cross-dataset generalisation

Train on Bohra, test on each HASOC set (romanised, clean per-tweet). The gap from the within-Bohra
F1 above is the generalisation drop.

In [13]:
h22_r = filter_romanised(h22, include_mixed=True)
print(summarise('Bohra -> HASOC22', evaluate_cross(bohra_r, h22_r)))

if h21 is not None:
    h21_r = filter_romanised(h21, include_mixed=True)
    if len(h21_r):
        print(summarise('Bohra -> HASOC21', evaluate_cross(bohra_r, h21_r)))

[Bohra -> HASOC22]  macro-F1 = 0.510   hate-F1 = 0.486   acc = 0.511   (train 4574, test 4306)
[Bohra -> HASOC21]  macro-F1 = 0.492   hate-F1 = 0.412   acc = 0.505   (train 4574, test 1831)


## 9. Save the unified corpus

In [14]:
out = '/content/drive/MyDrive/dissertation/data/unified_corpus.parquet'
corpus.to_parquet(out)
print('saved', len(corpus), 'rows ->', out)
print(dict(corpus['source'].value_counts()))

OSError: Cannot save file into a non-existent directory: '\content\drive\MyDrive\dissertation\data'

### Appendix - why not `final.csv`

`final.csv` glues growing thread context onto each row, which inflates and distorts the cross
number. Loaded separately here only for comparison; it is **not** used above.

In [ ]:
from hinglish_hate import load_hasoc2022
h22_finalcsv = load_hasoc2022(find_one(DATA_ROOT, 'final.csv'))
print(summarise('Bohra -> HASOC22 (final.csv, accumulated)',
                evaluate_cross(bohra_r, filter_romanised(h22_finalcsv, include_mixed=True))))
print(summarise('Bohra -> HASOC22 (clean per-tweet)      ',
                evaluate_cross(bohra_r, h22_r)))

### Notes
- `class_weight='balanced'` handles the hate/not imbalance; macro-F1 is the headline.
- Seeded (`random_state=42`) - reproducible.
- Keep the exact `bohra_r` / romanised frames so every later model (XLM-R, IndicBERT, MuRIL, LoRA
  LLM) is scored on identical splits.